<!-- notebook-header -->
# Deteccao de Objetos

**Modulo:** 05 - Dominios Aplicados / 05A - Computer Vision  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Bounding boxes, IoU, NMS, anchor boxes, YOLO, R-CNN e metricas de deteccao.


# Deteccao de Objetos

## Objetivo
Entender como modelos de deep learning localizam e classificam multiplos objetos em uma imagem,
desde conceitos fundamentais (bounding boxes, IoU) ate arquiteturas modernas (YOLO, Faster R-CNN).

## Pre-requisitos
- 5A_1 (CNN Fundamentos): convolucao, pooling, feature maps
- 5A_2 (Classificacao): transfer learning, training recipes

## Conteudo
1. Bounding Boxes e IoU
2. Two-Stage Detectors (R-CNN family)
3. One-Stage Detectors (YOLO, SSD, RetinaNet)
4. Anchor Boxes e Feature Pyramid Networks
5. Non-Maximum Suppression (NMS)
6. Data Augmentation para Deteccao
7. Exercicios Praticos
8. Erros Comuns e Armadilhas
9. Resumo e Conexoes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
np.random.seed(42)
print('Imports OK')

## 1. Bounding Boxes e IoU

### Analogia: Descrevendo a Posicao de Objetos

Imagine que voce esta ao telefone descrevendo onde esta um gato numa foto para alguem.
Voce poderia dizer: "o gato esta no canto inferior direito, ocupa uns 30% da imagem".
Bounding boxes fazem exatamente isso, mas com precisao matematica: um retangulo que
envolve o objeto.

### Definicao Formal

Existem dois formatos principais para representar bounding boxes:
- **Corner format** `[x1, y1, x2, y2]`: canto superior-esquerdo e canto inferior-direito
- **Center format** `[xc, yc, w, h]`: centro + largura + altura

A conversao entre formatos e trivial mas erros aqui sao a causa #1 de bugs em deteccao.

### Por que em ML: IoU como Metrica Universal

**Intersection over Union (IoU)** mede a qualidade de uma predicao:
- IoU = Area(Interseccao) / Area(Uniao)
- IoU = 1.0: predicao perfeita
- IoU = 0.0: nenhuma sobreposicao
- IoU >= 0.5: considerado "acerto" (threshold PASCAL VOC)
- COCO usa mAP@[0.5:0.95]: media de thresholds de 0.5 a 0.95 com step 0.05

In [ ]:

def box_iou_manual(box1, box2):
    # box1, box2: [x1, y1, x2, y2]
    x1_i = max(box1[0], box2[0])
    y1_i = max(box1[1], box2[1])
    x2_i = min(box1[2], box2[2])
    y2_i = min(box1[3], box2[3])
    
    if x2_i < x1_i or y2_i < y1_i:
        return 0.0
    
    intersection = (x2_i - x1_i) * (y2_i - y1_i)
    
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0

def format_conversion(boxes, from_format='corner', to_format='center'):
    # Convert between [x1,y1,x2,y2] and [xc,yc,w,h]
    if from_format == 'corner' and to_format == 'center':
        xc = (boxes[0] + boxes[2]) / 2
        yc = (boxes[1] + boxes[3]) / 2
        w = boxes[2] - boxes[0]
        h = boxes[3] - boxes[1]
        return [xc, yc, w, h]
    elif from_format == 'center' and to_format == 'corner':
        x1 = boxes[0] - boxes[2] / 2
        y1 = boxes[1] - boxes[3] / 2
        x2 = boxes[0] + boxes[2] / 2
        y2 = boxes[1] + boxes[3] / 2
        return [x1, y1, x2, y2]

# Exemplos
box_a = [50, 50, 150, 150]
box_b = [100, 100, 200, 200]
box_c = [200, 200, 300, 300]

iou_ab = box_iou_manual(box_a, box_b)
iou_ac = box_iou_manual(box_a, box_c)

print('Exemplos de IoU:')
print(f'IoU(box_a, box_b): {iou_ab:.4f}')
print(f'IoU(box_a, box_c): {iou_ac:.4f}')
print()

# Conversão de formatos
center_format = format_conversion(box_a, 'corner', 'center')
back_to_corner = format_conversion(center_format, 'center', 'corner')

print(f'Box A (corner): {box_a}')
print(f'Box A (center): {center_format}')
print(f'Back to corner: {[int(x) for x in back_to_corner]}')


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# High IoU
ax = axes[0]
ax.set_xlim(0, 300)
ax.set_ylim(0, 300)
ax.set_aspect('equal')
rect1 = patches.Rectangle((50, 50), 100, 100, linewidth=2, edgecolor='blue', facecolor='none', label='GT')
rect2 = patches.Rectangle((75, 75), 100, 100, linewidth=2, edgecolor='red', facecolor='none', label='Pred')
ax.add_patch(rect1)
ax.add_patch(rect2)
iou = box_iou_manual([50, 50, 150, 150], [75, 75, 175, 175])
ax.set_title(f'High IoU = {iou:.3f}')
ax.legend()
ax.grid(True, alpha=0.3)

# Medium IoU
ax = axes[1]
ax.set_xlim(0, 300)
ax.set_ylim(0, 300)
ax.set_aspect('equal')
rect1 = patches.Rectangle((50, 50), 100, 100, linewidth=2, edgecolor='blue', facecolor='none', label='GT')
rect2 = patches.Rectangle((120, 50), 80, 100, linewidth=2, edgecolor='red', facecolor='none', label='Pred')
ax.add_patch(rect1)
ax.add_patch(rect2)
iou = box_iou_manual([50, 50, 150, 150], [120, 50, 200, 150])
ax.set_title(f'Medium IoU = {iou:.3f}')
ax.legend()
ax.grid(True, alpha=0.3)

# Low IoU
ax = axes[2]
ax.set_xlim(0, 300)
ax.set_ylim(0, 300)
ax.set_aspect('equal')
rect1 = patches.Rectangle((50, 50), 100, 100, linewidth=2, edgecolor='blue', facecolor='none', label='GT')
rect2 = patches.Rectangle((180, 180), 80, 80, linewidth=2, edgecolor='red', facecolor='none', label='Pred')
ax.add_patch(rect1)
ax.add_patch(rect2)
iou = box_iou_manual([50, 50, 150, 150], [180, 180, 260, 260])
ax.set_title(f'Low IoU = {iou:.3f}')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/iou_examples.png', dpi=100, bbox_inches='tight')
plt.show()


### O que observar sobre IoU

1. IoU e **simetrico**: IoU(A,B) = IoU(B,A)
2. Mesmo uma predicao "quase certa" visualmente pode ter IoU baixo se o tamanho estiver errado
3. IoU = 0 nao significa que as boxes estao longe -- podem estar adjacentes sem sobreposicao
4. Para objetos muito pequenos (< 32x32 pixels), 1 pixel de erro reduz IoU drasticamente

### O que concluir sobre mAP

**mean Average Precision (mAP)** e a metrica padrao para deteccao:
1. Para cada classe, calcule a curva Precision-Recall
2. AP = area sob a curva PR (por classe)
3. mAP = media de AP sobre todas as classes

Na pratica, mAP@0.5 (PASCAL VOC) e mais permissivo, enquanto mAP@[0.5:0.95] (COCO)
penaliza predicoes imprecisas. Sempre reporte ambos para comparabilidade.

### Conexao com outros notebooks sobre Metricas

IoU conecta com 2_4 (metricas de avaliacao), onde vimos precision/recall para classificacao.
Em deteccao, precision = "das boxes preditas, quantas sao corretas?" e recall =
"dos objetos reais, quantos foram encontrados?" A curva PR e calculada por classe.

In [ ]:
# Demonstracao: como mAP e calculado
# Simular predicoes para 1 classe

def compute_ap(precisions, recalls):
    """Compute AP usando interpolacao 11-point (PASCAL VOC)."""
    ap = 0.0
    for t in np.linspace(0, 1, 11):
        prec_at_recall = [p for p, r in zip(precisions, recalls) if r >= t]
        if prec_at_recall:
            ap += max(prec_at_recall) / 11.0
    return ap

# Simular: 10 predicoes ordenadas por confidence
np.random.seed(42)
n_pred = 10
n_gt = 5  # 5 objetos reais

# Cada predicao: True Positive ou False Positive
# (em ordem decrescente de confidence)
tp_fp = [1, 1, 0, 1, 0, 0, 1, 0, 1, 0]  # 1=TP, 0=FP

# Calcular precision e recall cumulativos
precisions = []
recalls = []
tp_cum = 0
fp_cum = 0
for i, is_tp in enumerate(tp_fp):
    if is_tp:
        tp_cum += 1
    else:
        fp_cum += 1
    precisions.append(tp_cum / (tp_cum + fp_cum))
    recalls.append(tp_cum / n_gt)

ap = compute_ap(precisions, recalls)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision-Recall curve
ax = axes[0]
ax.plot(recalls, precisions, 'b-o', linewidth=2, markersize=6)
ax.fill_between(recalls, precisions, alpha=0.2)
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title(f'Curva Precision-Recall (AP = {ap:.3f})', fontsize=13, fontweight='bold')
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

# Annotate each point
for i, (r, p) in enumerate(zip(recalls, precisions)):
    label = 'TP' if tp_fp[i] else 'FP'
    color = 'green' if tp_fp[i] else 'red'
    ax.annotate(f'{label}', (r, p), textcoords="offset points",
                xytext=(5, 5), fontsize=8, color=color)

# Thresholds comparison
ax = axes[1]
thresholds = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
# Simular AP diminuindo com threshold mais rigoroso
aps = [0.82, 0.78, 0.73, 0.67, 0.60, 0.52, 0.41, 0.30, 0.18, 0.08]
ax.bar(range(len(thresholds)), aps, color='steelblue', alpha=0.7)
ax.set_xticks(range(len(thresholds)))
ax.set_xticklabels([f'{t:.2f}' for t in thresholds], rotation=45)
ax.set_xlabel('IoU Threshold', fontsize=12)
ax.set_ylabel('AP', fontsize=12)
ax.set_title(f'AP por Threshold (mAP@[.5:.95] = {np.mean(aps):.3f})', fontsize=13, fontweight='bold')
ax.axhline(y=np.mean(aps), color='red', linestyle='--', label=f'mAP = {np.mean(aps):.3f}')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/map_calculation.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'AP@0.5 = {aps[0]:.3f} (permissivo -- PASCAL VOC)')
print(f'AP@0.75 = {aps[5]:.3f} (rigoroso)')
print(f'mAP@[0.5:0.95] = {np.mean(aps):.3f} (padrao COCO)')

## 2. Two-Stage Detectors (R-CNN Family)

### Analogia: Busca em Dois Passos

Imagine procurar um livro numa biblioteca enorme:
- **Passo 1 (Region Proposal):** caminhe pelos corredores e marque prateleiras que *parecem* relevantes
- **Passo 2 (Classification):** examine cada prateleira marcada com cuidado

Two-stage detectors fazem exatamente isso: primeiro geram "propostas de regiao" (onde *pode* haver
objetos), depois classificam e refinam cada proposta.

### Definicao Formal: Evolucao da Familia R-CNN

| Modelo | Ano | Region Proposals | Feature Extraction | Velocidade |
|--------|-----|------------------|--------------------|------------|
| R-CNN | 2014 | Selective Search (2000 regioes) | CNN por regiao | ~47s/img |
| Fast R-CNN | 2015 | Selective Search | CNN compartilhada + RoI Pooling | ~2s/img |
| Faster R-CNN | 2016 | RPN (aprendido) | CNN compartilhada + RoI Pooling | ~0.2s/img |

A evolucao chave: mover o bottleneck de "computacao repetida" para "computacao compartilhada".

### Por que em ML: Faster R-CNN como Baseline

Faster R-CNN ainda e o baseline mais usado em papers academicos por sua modularidade:
backbone intercambiavel, head flexivel, e performance consistente. Quando papers reportam
"state-of-the-art", geralmente comparam contra Faster R-CNN + FPN.

In [ ]:
# Visualizacao: Evolucao da familia R-CNN
fig, ax = plt.subplots(figsize=(14, 6))

# Timeline dos modelos
models = [
    ('R-CNN\n(2014)', 2014, 47.0, 58.5, 'Selective Search\n+ CNN per region'),
    ('SPPNet\n(2014)', 2014.5, 2.3, 59.2, 'Spatial Pyramid\nPooling'),
    ('Fast R-CNN\n(2015)', 2015, 2.0, 66.9, 'RoI Pooling\nshared features'),
    ('Faster R-CNN\n(2016)', 2016, 0.2, 73.2, 'RPN (learned\nproposals)'),
    ('Mask R-CNN\n(2017)', 2017, 0.2, 78.3, '+ Instance\nSegmentation'),
    ('Cascade R-CNN\n(2018)', 2018, 0.15, 82.1, 'Multi-stage\nrefinement'),
]

years = [m[1] for m in models]
speeds = [m[2] for m in models]
maps = [m[3] for m in models]
names = [m[0] for m in models]
notes = [m[4] for m in models]

# Plot mAP progression
ax.plot(years, maps, 'bo-', linewidth=2, markersize=10, zorder=5)

for i, (name, year, speed, mAP, note) in enumerate(models):
    # Annotate name + speed
    offset_y = 2.5 if i % 2 == 0 else -4
    ax.annotate(f'{name}\nmAP={mAP}', (year, mAP),
                textcoords="offset points", xytext=(0, 15),
                fontsize=9, ha='center', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
    # Speed annotation below
    ax.annotate(f'{speed}s/img', (year, mAP),
                textcoords="offset points", xytext=(0, -18),
                fontsize=8, ha='center', color='gray')

ax.set_xlabel('Ano', fontsize=12)
ax.set_ylabel('mAP (COCO)', fontsize=12)
ax.set_title('Evolucao dos Two-Stage Detectors', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim(50, 90)

plt.tight_layout()
plt.savefig('/tmp/rcnn_evolution.png', dpi=100, bbox_inches='tight')
plt.show()

print('Insight chave: a cada geracao, velocidade aumentou 10-200x mantendo accuracy')
print('O salto critico foi Faster R-CNN: RPN eliminou Selective Search (bottleneck)')

### O que observar sobre RoI Pooling e RoI Align

**RoI Pooling** (Fast R-CNN) discretiza regioes para tamanho fixo (ex: 7x7), mas causa
desalinhamento espacial (quantization error). **RoI Align** (Mask R-CNN) usa interpolacao
bilinear, eliminando esse erro. Isso e critico para segmentacao pixel-level mas tambem
melhora deteccao em ~1-2% mAP.

### O que concluir sobre Quando Usar Two-Stage

Two-stage detectors ainda sao a melhor escolha quando:
- Accuracy e mais importante que velocidade (diagnostico medico, satelite)
- Objetos sao muito pequenos ou muito proximos
- Voce precisa de instance segmentation (Mask R-CNN)

Para aplicacoes real-time (video, robotica), one-stage detectors sao preferidos.

### Conexao com outros notebooks sobre Feature Extraction

O backbone do Faster R-CNN (ResNet + FPN) usa exatamente os conceitos de 5A_1:
convolucao hierarquica, skip connections, e feature maps multi-escala.

## 3. One-Stage Detectors (YOLO, SSD, RetinaNet)

### Analogia: Olhar Tudo de Uma Vez

Se two-stage e como "procurar prateleiras relevantes e depois examinar", one-stage e como
**olhar para a biblioteca inteira de uma vez** e identificar todos os livros simultaneamente.
E menos preciso, mas muito mais rapido.

### Definicao Formal: YOLO

YOLO (You Only Look Once) divide a imagem em um grid SxS. Cada celula do grid prediz:
- B bounding boxes (cada com 4 coordenadas + 1 confidence)
- C probabilidades de classe

Total de saida: S x S x (B * 5 + C)

### Por que em ML: YOLO como Padrao para Real-Time

YOLO domina aplicacoes real-time: cameras de seguranca, carros autonomos, drones,
robotica. YOLOv8 (2023) e anchor-free e atinge >50 FPS em GPU consumer com mAP competitivo.

In [ ]:

# Visualizar conceito de YOLO
def visualize_yolo_grid(image_size=416, grid_size=13, num_classes=80, num_boxes=5):
    # YOLO divide a imagem em grid_size x grid_size
    # Cada célula prediz:
    # - num_boxes bounding boxes (4 coords + 1 confidence)
    # - num_classes class probabilities
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Grid visualization
    ax = axes[0]
    ax.set_xlim(0, image_size)
    ax.set_ylim(0, image_size)
    ax.set_aspect('equal')
    
    cell_size = image_size // grid_size
    
    # Draw grid
    for i in range(grid_size + 1):
        ax.axhline(y=i * cell_size, color='gray', linewidth=0.5)
        ax.axvline(x=i * cell_size, color='gray', linewidth=0.5)
    
    # Exemplo: 2 objetos
    # Objeto 1 na célula (3, 5)
    obj1_cell = (3, 5)
    obj1_rect = patches.Rectangle((obj1_cell[0] * cell_size, obj1_cell[1] * cell_size),
                                   cell_size, cell_size, linewidth=2,
                                   edgecolor='red', facecolor='red', alpha=0.3)
    ax.add_patch(obj1_rect)
    ax.text(obj1_cell[0] * cell_size + 5, obj1_cell[1] * cell_size + 15, 'Cat', color='red')
    
    # Objeto 2 na célula (8, 9)
    obj2_cell = (8, 9)
    obj2_rect = patches.Rectangle((obj2_cell[0] * cell_size, obj2_cell[1] * cell_size),
                                   cell_size, cell_size, linewidth=2,
                                   edgecolor='blue', facecolor='blue', alpha=0.3)
    ax.add_patch(obj2_rect)
    ax.text(obj2_cell[0] * cell_size + 5, obj2_cell[1] * cell_size + 15, 'Dog', color='blue')
    
    ax.set_title(f'YOLO Grid ({grid_size}x{grid_size})')
    ax.set_xlabel('Width')
    ax.set_ylabel('Height')
    
    # Output tensor structure
    ax = axes[1]
    ax.axis('off')
    
    structure_text = f'''YOLO Output Structure:
    
Tensor Shape: (batch, {grid_size}x{grid_size}, {num_boxes}*(5+{num_classes}))

Per Cell Prediction:
  - {num_boxes} bounding boxes
  - Each box: [x, y, w, h, confidence]
  - {num_classes} class probabilities
  
Total output size:
  {grid_size} x {grid_size} x {num_boxes*(5+num_classes)}
  = {grid_size * grid_size * num_boxes * (5 + num_classes)} values per image
  
Interpretação:
  - (x, y): Center offset dentro da célula
  - (w, h): Width/height (normalized)
  - confidence: P(object) * IoU(pred, truth)
  - class probs: P(class | object)
'''
    
    ax.text(0.05, 0.95, structure_text, transform=ax.transAxes,
           fontsize=10, verticalalignment='top', family='monospace',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.savefig('/tmp/yolo_concept.png', dpi=100, bbox_inches='tight')
    plt.show()

visualize_yolo_grid()


In [ ]:
# Comparacao: One-Stage vs Two-Stage
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Speed vs Accuracy scatter
detectors = {
    'Two-Stage': [
        ('Faster R-CNN', 15, 42.0),
        ('Cascade R-CNN', 12, 44.3),
        ('Mask R-CNN', 13, 43.1),
    ],
    'One-Stage': [
        ('YOLOv5-S', 120, 37.4),
        ('YOLOv5-M', 80, 45.4),
        ('YOLOv8-M', 90, 50.2),
        ('SSD-300', 46, 25.1),
        ('RetinaNet', 25, 40.1),
        ('FCOS', 30, 42.1),
    ]
}

ax = axes[0]
for category, models_list in detectors.items():
    names = [m[0] for m in models_list]
    fps_vals = [m[1] for m in models_list]
    mAP_vals = [m[2] for m in models_list]
    color = 'blue' if category == 'Two-Stage' else 'red'
    marker = 's' if category == 'Two-Stage' else 'o'
    ax.scatter(fps_vals, mAP_vals, c=color, marker=marker, s=100,
              label=category, zorder=5)
    for name, fps, mAP in models_list:
        ax.annotate(name, (fps, mAP), textcoords="offset points",
                   xytext=(5, 5), fontsize=8)

ax.set_xlabel('FPS (frames por segundo)', fontsize=12)
ax.set_ylabel('mAP@0.5 (COCO)', fontsize=12)
ax.set_title('Speed vs Accuracy: Two-Stage vs One-Stage', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# YOLO evolution
ax = axes[1]
yolo_versions = ['v1\n(2016)', 'v2\n(2017)', 'v3\n(2018)', 'v4\n(2020)',
                 'v5\n(2020)', 'v7\n(2022)', 'v8\n(2023)']
yolo_maps = [63.4, 76.8, 82.3, 83.0, 85.5, 87.2, 88.9]  # mAP@0.5 COCO

bars = ax.bar(range(len(yolo_versions)), yolo_maps, color='orangered', alpha=0.7)
ax.set_xticks(range(len(yolo_versions)))
ax.set_xticklabels(yolo_versions)
ax.set_ylabel('mAP@0.5 (COCO)', fontsize=12)
ax.set_title('Evolucao do YOLO', fontsize=13, fontweight='bold')
ax.set_ylim(55, 95)
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, yolo_maps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/detector_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print('Conclusao: YOLO domina real-time, Faster R-CNN domina accuracy-first')
print('RetinaNet com Focal Loss fechou a gap de accuracy entre one-stage e two-stage')

### O que observar sobre Focal Loss (RetinaNet)

O problema fundamental de one-stage detectors: a maioria das celulas do grid NAO contem
objetos (background). Isso cria um **class imbalance extremo** (1:1000 positivo:negativo).

**Focal Loss** resolve isso reduzindo o peso dos exemplos faceis (background):
- FL(pt) = -alpha * (1 - pt)^gamma * log(pt)
- gamma = 2: exemplos faceis com pt > 0.5 tem peso quase zero
- Resultado: o modelo foca nos exemplos dificeis (objetos reais)

### O que concluir sobre Anchor-Free Detectors

A tendencia moderna e **anchor-free**: em vez de predefinir formatos de boxes,
o modelo prediz diretamente o centro do objeto + distancias ate as bordas.
Exemplos: FCOS, CenterNet, YOLOv8. Vantagens: sem hiperparametros de anchor,
treino mais simples, melhor para objetos de formas incomuns.

### Conexao com outros notebooks sobre Loss Functions

Focal Loss conecta com 5A_2 (loss functions para classificacao), onde vimos Cross-Entropy
e suas variantes. A intuicao e a mesma: ponderar exemplos para lidar com imbalance.

## 4. Anchor Boxes e Feature Pyramid Networks

### Analogia: Armadilhas de Tamanhos Diferentes

Imagine colocar armadilhas numa floresta para capturar animais:
- Armadilha pequena (32x32): captura passaros
- Armadilha media (128x128): captura coelhos
- Armadilha grande (512x512): captura cervos

**Anchor boxes** sao "armadilhas pre-definidas" com diferentes tamanhos e proporcoes.
O modelo aprende a **ajustar** cada anchor para encaixar no objeto real (regressao de bbox).

### Definicao Formal: FPN

**Feature Pyramid Network (FPN)** resolve o problema de escala:
- Backbone (ResNet) produz feature maps em multiplas resolucoes: C2 (1/4), C3 (1/8), C4 (1/16), C5 (1/32)
- FPN cria um caminho **top-down**: P5 -> P4 -> P3 -> P2
- Cada nivel Pi detecta objetos de tamanho diferente
- Lateral connections preservam features de alta resolucao

### Por que em ML: Multi-Scale Detection

Objetos no mundo real variam enormemente de tamanho. Uma camera de seguranca ve
pessoas perto (grande) e longe (pequena) simultaneamente. FPN permite detectar
ambos com a mesma rede, sem precisar processar a imagem em multiplas escalas.

In [ ]:
# Conceito de Anchor Boxes
def generate_anchors(scales=[0.5, 1.0, 2.0], ratios=[0.5, 1.0, 2.0]):
    anchors = []
    for s in scales:
        for r in ratios:
            w = s * np.sqrt(1 / r)
            h = s * np.sqrt(r)
            anchors.append([w, h])
    return np.array(anchors)

anchors = generate_anchors()

print('Anchor Boxes (Normalized):')
print(f"{'Scale':<10} {'Ratio':<10} {'Width':<10} {'Height':<10}")
print('-' * 40)

idx = 0
for s in [0.5, 1.0, 2.0]:
    for r in [0.5, 1.0, 2.0]:
        w, h = anchors[idx]
        print(f'{s:<10} {r:<10} {w:<10.3f} {h:<10.3f}')
        idx += 1

# Visualizar
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)

colors = plt.cm.tab10(np.linspace(0, 1, len(anchors)))

for i, (w, h) in enumerate(anchors):
    rect = patches.Rectangle((-w/2, -h/2), w, h, linewidth=2,
                             edgecolor=colors[i], facecolor='none')
    ax.add_patch(rect)
    ax.text(-w/2, -h/2, str(i), fontsize=8)

ax.set_xlabel('Width')
ax.set_ylabel('Height')
ax.set_title('Anchor Boxes (9 combinacoes de 3 escalas x 3 ratios)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/anchor_boxes.png', dpi=100, bbox_inches='tight')
plt.show()

print()
print('9 anchors por posicao e o padrao (Faster R-CNN)')
print('K-means nos aspect ratios do dataset gera anchors melhores (YOLOv2)')

In [ ]:
# Visualizacao: Feature Pyramid Network
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# FPN Architecture diagram
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('Feature Pyramid Network (FPN)', fontsize=14, fontweight='bold')

# Bottom-up pathway (backbone)
backbone_levels = [
    ('C2 56x56', 1, 1, 1.5, 1.5, '#4CAF50'),
    ('C3 28x28', 1, 3, 1.2, 1.2, '#2196F3'),
    ('C4 14x14', 1, 5.5, 0.9, 0.9, '#FF9800'),
    ('C5 7x7',   1, 7.5, 0.6, 0.6, '#F44336'),
]

for label, x, y, w, h, color in backbone_levels:
    rect = patches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                                   facecolor=color, alpha=0.6, edgecolor='black')
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=8, fontweight='bold')

# Arrows bottom-up
for i in range(len(backbone_levels) - 1):
    y_start = backbone_levels[i][2] + backbone_levels[i][4]
    y_end = backbone_levels[i+1][2]
    ax.annotate('', xy=(1.5, y_end), xytext=(1.5, y_start),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

ax.text(1.5, 10, 'Bottom-Up\n(Backbone)', ha='center', fontsize=10, fontweight='bold')

# Top-down pathway
fpn_levels = [
    ('P2 56x56', 5, 1, 1.5, 1.5, '#4CAF50'),
    ('P3 28x28', 5, 3, 1.2, 1.2, '#2196F3'),
    ('P4 14x14', 5, 5.5, 0.9, 0.9, '#FF9800'),
    ('P5 7x7',   5, 7.5, 0.6, 0.6, '#F44336'),
]

for label, x, y, w, h, color in fpn_levels:
    rect = patches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                                   facecolor=color, alpha=0.3, edgecolor='black', linestyle='--')
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=8, fontweight='bold')

# Arrows top-down
for i in range(len(fpn_levels) - 1, 0, -1):
    ax.annotate('', xy=(5.5, fpn_levels[i-1][2] + fpn_levels[i-1][4]),
                xytext=(5.5, fpn_levels[i][2]),
                arrowprops=dict(arrowstyle='->', color='red', lw=2))

# Lateral connections
for i in range(len(backbone_levels)):
    bx = backbone_levels[i][1] + backbone_levels[i][3]
    by = backbone_levels[i][2] + backbone_levels[i][4]/2
    fx = fpn_levels[i][1]
    fy = fpn_levels[i][2] + fpn_levels[i][4]/2
    ax.annotate('', xy=(fx, fy), xytext=(bx, by),
                arrowprops=dict(arrowstyle='->', color='blue', lw=1.5, linestyle='--'))

ax.text(5.5, 10, 'Top-Down\n+ Lateral', ha='center', fontsize=10, fontweight='bold')

# Which level detects which size
ax = axes[1]
ax.axis('off')
ax.set_title('Qual Nivel Detecta Qual Tamanho?', fontsize=14, fontweight='bold')

detect_info = [
    ('P2 (56x56)', 'Objetos PEQUENOS (< 32px)', '#4CAF50',
     'pedestres distantes, sinais pequenos'),
    ('P3 (28x28)', 'Objetos MEDIOS (32-96px)', '#2196F3',
     'carros, pessoas em distancia media'),
    ('P4 (14x14)', 'Objetos GRANDES (96-256px)', '#FF9800',
     'veiculos proximos, placas grandes'),
    ('P5 (7x7)', 'Objetos MUITO GRANDES (>256px)', '#F44336',
     'onibus, caminhoes, objetos proximos'),
]

for i, (level, size_range, color, examples) in enumerate(detect_info):
    y = 0.85 - i * 0.22
    ax.text(0.05, y, level, fontsize=12, fontweight='bold', color=color,
            transform=ax.transAxes)
    ax.text(0.30, y, size_range, fontsize=10, transform=ax.transAxes)
    ax.text(0.05, y - 0.06, f'Ex: {examples}', fontsize=9, color='gray',
            transform=ax.transAxes)

plt.tight_layout()
plt.savefig('/tmp/fpn_architecture.png', dpi=100, bbox_inches='tight')
plt.show()

print('FPN resolve o maior desafio de deteccao: objetos de tamanhos variados')
print('Sem FPN, detectores sao bons para um tamanho mas falham para outros')

### O que observar sobre Anchor Design

O design dos anchors e crucial para performance:
- **Escalas:** tipicamente 3 (32, 64, 128 pixels ou ratios 0.5, 1.0, 2.0)
- **Ratios:** tipicamente 3 (1:2, 1:1, 2:1 para objetos altos, quadrados, largos)
- 9 anchors por posicao e o padrao (3 escalas x 3 ratios)
- K-means no dataset de treino pode gerar anchors melhores (YOLOv2 fez isso)

### O que concluir sobre a Importancia do FPN

Antes do FPN, detectores usavam apenas o ultimo feature map (C5, 7x7).
Isso e terrivel para objetos pequenos porque a resolucao e muito baixa.
FPN resolveu isso e se tornou componente obrigatorio: Faster R-CNN + FPN,
RetinaNet + FPN, Mask R-CNN + FPN. A melhoria tipica e +3-5% mAP.

### Conexao com outros notebooks sobre Multi-Scale

FPN conecta com 5A_1 (CNN fundamentos), onde vimos que camadas iniciais capturam
detalhes (bordas, texturas) e camadas profundas capturam semantica (objetos).
FPN explora exatamente essa hierarquia para deteccao multi-escala.

## 5. Non-Maximum Suppression (NMS)

### Analogia: Eliminando Duplicatas

Voce pediu a 3 amigos para marcar onde esta o gato na foto. Cada um desenhou um
retangulo ligeiramente diferente. NMS e o processo de "ok, todos marcaram mais ou menos
o mesmo lugar, vamos ficar com a marcacao mais confiante e descartar as duplicatas."

### Definicao Formal: Algoritmo NMS

1. Ordenar predicoes por confidence (decrescente)
2. Selecionar a predicao com maior confidence -> adicionar a "mantidos"
3. Calcular IoU entre ela e todas as restantes
4. Remover predicoes com IoU > threshold (sao duplicatas)
5. Repetir ate nao sobrar nenhuma predicao

### Por que em ML: NMS e Indispensavel

Sem NMS, um detector tipico produz 5-50 boxes por objeto. NMS reduz para ~1 por objeto.
Variantes modernas (Soft-NMS, DIoU-NMS) melhoram para cenarios com objetos proximos.

In [ ]:

def nms_manual(boxes, confidences, iou_threshold=0.5):
    # boxes: [[x1, y1, x2, y2], ...] (corner format)
    # confidences: [score1, score2, ...]
    
    if len(boxes) == 0:
        return []
    
    # Sort by confidence (descending)
    sorted_idx = np.argsort(confidences)[::-1]
    
    keep = []
    while len(sorted_idx) > 0:
        # Select box with highest confidence
        current = sorted_idx[0]
        keep.append(current)
        
        if len(sorted_idx) == 1:
            break
        
        # Calculate IoU with remaining boxes
        ious = []
        current_box = boxes[current]
        for i in sorted_idx[1:]:
            iou = box_iou_manual(current_box, boxes[i])
            ious.append(iou)
        
        # Keep only boxes with IoU < threshold
        ious = np.array(ious)
        sorted_idx = sorted_idx[1:][ious < iou_threshold]
    
    return keep

# Exemplo
boxes = [
    [10, 10, 100, 100],
    [20, 20, 110, 110],
    [15, 15, 105, 105],
    [200, 200, 300, 300],
    [210, 210, 310, 310]
]

confidences = [0.95, 0.87, 0.80, 0.92, 0.88]

keep_idx = nms_manual(boxes, confidences, iou_threshold=0.5)

print('NMS Results:')
print(f'Input boxes: {len(boxes)}')
print(f'Output boxes: {len(keep_idx)}')
print(f'Kept indices: {keep_idx}')
print()
print('Kept boxes:')
for idx in keep_idx:
    print(f'  Box {idx}: {boxes[idx]}, confidence: {confidences[idx]:.2f}')


In [ ]:
# Visualizacao: NMS em acao
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Cenario: 2 objetos reais, 8 predicoes
np.random.seed(42)

# Ground truth
gt_boxes = [[30, 30, 130, 130], [200, 180, 300, 280]]

# Predicoes (varias para cada objeto + 2 falsos positivos)
pred_boxes = [
    [28, 28, 128, 128],   # perto de GT1, conf=0.95
    [35, 35, 135, 135],   # perto de GT1, conf=0.87
    [25, 32, 125, 132],   # perto de GT1, conf=0.80
    [198, 178, 298, 278], # perto de GT2, conf=0.92
    [205, 185, 305, 285], # perto de GT2, conf=0.85
    [300, 50, 350, 100],  # falso positivo, conf=0.45
    [32, 180, 80, 230],   # falso positivo, conf=0.30
    [40, 40, 140, 140],   # perto de GT1, conf=0.70
]
pred_confs = [0.95, 0.87, 0.80, 0.92, 0.85, 0.45, 0.30, 0.70]

# Plot 1: Antes do NMS
ax = axes[0]
ax.set_xlim(0, 370)
ax.set_ylim(0, 320)
ax.set_aspect('equal')
ax.set_title(f'Antes do NMS ({len(pred_boxes)} boxes)', fontsize=12, fontweight='bold')

for gt in gt_boxes:
    r = patches.Rectangle((gt[0], gt[1]), gt[2]-gt[0], gt[3]-gt[1],
                          linewidth=3, edgecolor='green', facecolor='green', alpha=0.15)
    ax.add_patch(r)

for box, conf in zip(pred_boxes, pred_confs):
    alpha = max(0.3, conf)
    r = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                          linewidth=1.5, edgecolor='red', facecolor='none', alpha=alpha)
    ax.add_patch(r)
    ax.text(box[0], box[1]-3, f'{conf:.2f}', fontsize=7, color='red')

ax.grid(True, alpha=0.2)

# Aplicar confidence threshold
conf_threshold = 0.5
filtered_idx = [i for i, c in enumerate(pred_confs) if c >= conf_threshold]
filtered_boxes = [pred_boxes[i] for i in filtered_idx]
filtered_confs = [pred_confs[i] for i in filtered_idx]

# Plot 2: Apos confidence threshold
ax = axes[1]
ax.set_xlim(0, 370)
ax.set_ylim(0, 320)
ax.set_aspect('equal')
ax.set_title(f'Confidence > {conf_threshold} ({len(filtered_boxes)} boxes)', fontsize=12, fontweight='bold')

for gt in gt_boxes:
    r = patches.Rectangle((gt[0], gt[1]), gt[2]-gt[0], gt[3]-gt[1],
                          linewidth=3, edgecolor='green', facecolor='green', alpha=0.15)
    ax.add_patch(r)

for box, conf in zip(filtered_boxes, filtered_confs):
    r = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                          linewidth=1.5, edgecolor='orange', facecolor='none')
    ax.add_patch(r)
    ax.text(box[0], box[1]-3, f'{conf:.2f}', fontsize=7, color='orange')

ax.grid(True, alpha=0.2)

# Plot 3: Apos NMS
keep_idx = nms_manual(filtered_boxes, filtered_confs, iou_threshold=0.5)
final_boxes = [filtered_boxes[i] for i in keep_idx]
final_confs = [filtered_confs[i] for i in keep_idx]

ax = axes[2]
ax.set_xlim(0, 370)
ax.set_ylim(0, 320)
ax.set_aspect('equal')
ax.set_title(f'Apos NMS ({len(final_boxes)} boxes)', fontsize=12, fontweight='bold')

for gt in gt_boxes:
    r = patches.Rectangle((gt[0], gt[1]), gt[2]-gt[0], gt[3]-gt[1],
                          linewidth=3, edgecolor='green', facecolor='green', alpha=0.15)
    ax.add_patch(r)

colors_final = ['blue', 'purple', 'cyan']
for i, (box, conf) in enumerate(zip(final_boxes, final_confs)):
    c = colors_final[i % len(colors_final)]
    r = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                          linewidth=3, edgecolor=c, facecolor=c, alpha=0.2)
    ax.add_patch(r)
    ax.text(box[0], box[1]-3, f'{conf:.2f}', fontsize=9, color=c, fontweight='bold')

ax.grid(True, alpha=0.2)

plt.suptitle('Pipeline: Predicoes -> Confidence Threshold -> NMS', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/nms_pipeline.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Pipeline completo:')
print(f'  {len(pred_boxes)} predicoes -> {len(filtered_boxes)} apos threshold -> {len(final_boxes)} apos NMS')
print(f'  NMS removeu {len(filtered_boxes) - len(final_boxes)} duplicatas')

### O que observar sobre Variantes de NMS

| Variante | Comportamento | Quando usar |
|----------|--------------|-------------|
| Hard NMS | Remove boxes com IoU > threshold | Default, funciona bem geralmente |
| Soft-NMS | Reduz score em vez de remover | Objetos proximos/sobrepostos |
| DIoU-NMS | Usa distancia dos centros + IoU | Objetos de tamanhos diferentes |
| Matrix NMS | NMS paralelo em GPU | Deploy com alta velocidade |

### O que concluir sobre o Threshold de NMS

O threshold de NMS e um trade-off:
- **Baixo (0.3):** remove mais duplicatas, mas pode remover objetos proximos validos
- **Alto (0.7):** mantém objetos proximos, mas pode deixar duplicatas

Na pratica, 0.5 e o default seguro. Para crowds (muitas pessoas juntas), use 0.6-0.7.

### Conexao com outros notebooks sobre Post-Processing

NMS e uma etapa de **pos-processamento**. Conecta com 3_5 (clustering) -- NMS e
essencialmente um tipo de clustering greedy: agrupar predicoes similares e manter
o representante mais confiante de cada grupo.

## 6. Data Augmentation para Deteccao

### Analogia: Fotografo Criativo

Um fotografo tirando fotos para um dataset de treino nao pode ir a todos os lugares
e cenarios possiveis. Data augmentation e como se o fotografo tivesse Photoshop e
pudesse criar variacoes infinitas: mudar iluminacao, recortar, espelhar, combinar fotos.

### Definicao Formal: Augmentations que Preservam Boxes

Em deteccao, augmentation e mais complexo que em classificacao porque **os bounding boxes
precisam ser transformados junto com a imagem**:
- Flip horizontal: espelhar coordenadas x
- Crop: recalcular boxes (e remover objetos cortados)
- Resize: escalar coordenadas proporcionalmente
- Mosaic (YOLOv4+): combinar 4 imagens em 1

### Por que em ML: Mosaic como Game-Changer

Mosaic augmentation combina 4 imagens em uma, criando contextos artificiais que o modelo
nunca veria naturalmente. Isso ajuda especialmente em objetos pequenos (que agora aparecem
em diferentes posicoes) e reduz a necessidade de batch normalization com batches grandes.

In [ ]:
# Demonstracao: Data Augmentation para Deteccao
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

np.random.seed(42)

# Imagem original simulada (background + objeto)
def create_scene(ax, title, boxes, labels, transform_name=''):
    ax.set_xlim(0, 200)
    ax.set_ylim(0, 200)
    ax.set_aspect('equal')
    ax.set_facecolor('#f0f0f0')

    # Background elements
    for _ in range(5):
        x, y = np.random.randint(0, 180), np.random.randint(0, 180)
        circle = patches.Circle((x, y), 5, facecolor='lightgreen', alpha=0.3)
        ax.add_patch(circle)

    colors = ['red', 'blue', 'orange']
    for i, (box, label) in enumerate(zip(boxes, labels)):
        c = colors[i % len(colors)]
        rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                                linewidth=2, edgecolor=c, facecolor=c, alpha=0.3)
        ax.add_patch(rect)
        ax.text(box[0]+2, box[1]+12, label, fontsize=9, color=c, fontweight='bold')

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.2)

# 1. Original
boxes_orig = [[30, 40, 90, 100], [120, 80, 180, 160]]
labels_orig = ['gato', 'carro']
create_scene(axes[0,0], 'Original', boxes_orig, labels_orig)

# 2. Horizontal Flip
boxes_flip = [[200-90, 40, 200-30, 100], [200-180, 80, 200-120, 160]]
create_scene(axes[0,1], 'Horizontal Flip', boxes_flip, labels_orig)
axes[0,1].invert_xaxis()

# 3. Random Crop (zoom in)
boxes_crop = [[10, 20, 70, 80], [100, 60, 160, 140]]
create_scene(axes[0,2], 'Random Crop + Resize', boxes_crop, labels_orig)

# 4. Color Jitter (simulated)
create_scene(axes[1,0], 'Color Jitter', boxes_orig, labels_orig)
axes[1,0].set_facecolor('#ffe0b2')  # warmer tone

# 5. Mosaic (4 imagens em 1)
ax = axes[1,1]
ax.set_xlim(0, 200)
ax.set_ylim(0, 200)
ax.set_aspect('equal')
ax.axhline(y=100, color='white', linewidth=3)
ax.axvline(x=100, color='white', linewidth=3)

# 4 quadrantes com diferentes objetos
quadrant_colors = ['#ffcdd2', '#c8e6c9', '#bbdefb', '#fff9c4']
for qi, (qx, qy) in enumerate([(0,100), (100,100), (0,0), (100,0)]):
    rect = patches.Rectangle((qx, qy), 100, 100, facecolor=quadrant_colors[qi], alpha=0.5)
    ax.add_patch(rect)
    # Small object in each quadrant
    ox = qx + np.random.randint(10, 60)
    oy = qy + np.random.randint(10, 60)
    obj = patches.Rectangle((ox, oy), 30, 25, linewidth=2,
                            edgecolor='red', facecolor='red', alpha=0.3)
    ax.add_patch(obj)

ax.set_title('Mosaic (4-in-1)', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.2)

# 6. Copy-Paste
create_scene(axes[1,2], 'Copy-Paste', boxes_orig + [[50, 140, 110, 190]], labels_orig + ['gato*'])
axes[1,2].annotate('copiado!', xy=(80, 165), fontsize=8, color='purple',
                   arrowprops=dict(arrowstyle='->', color='purple'),
                   xytext=(130, 180))

plt.suptitle('Data Augmentation para Deteccao de Objetos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/detection_augmentation.png', dpi=100, bbox_inches='tight')
plt.show()

print('Augmentations para deteccao DEVEM transformar as bounding boxes junto!')
print('Mosaic e o augmentation mais impactante para deteccao (YOLOv4+)')

### O que observar sobre Augmentation para Objetos Pequenos

Objetos pequenos (< 32x32 pixels) sao o calcanhar de Aquiles da deteccao.
Augmentations que ajudam: Mosaic (objetos aparecem em diferentes contextos),
Copy-Paste (multiplicar objetos raros), e Multi-Scale Training (treinar com
imagens de diferentes resolucoes). COCO dedica mAP-small como metrica separada.

### O que concluir sobre o Impacto do Augmentation

Na pratica, augmentation pode melhorar mAP em +5-10%, especialmente com datasets
pequenos (< 10K imagens). O combo mais eficaz para deteccao e:
Mosaic + Color Jitter + Random Scale + Horizontal Flip.

### Conexao com outros notebooks sobre Data Augmentation

Augmentation para deteccao estende 5A_2 (augmentation para classificacao). A diferenca
fundamental e que em classificacao, transformamos apenas a imagem; em deteccao,
transformamos imagem + boxes + labels simultaneamente.

## 7. Exercicios Praticos

### Exercicio 1: Implementar Soft-NMS
Soft-NMS nao remove boxes duplicadas -- em vez disso, reduz o score proporcionalmente
ao IoU. Isso e melhor para cenarios com objetos sobrepostos (ex: multidao).

Faca o seguinte:
- Implemente `soft_nms(boxes, scores, sigma=0.5, threshold=0.001)`
- Em vez de remover, aplique: score = score * exp(-IoU^2 / sigma)
- Compare resultado com Hard NMS

In [ ]:
# PRATICA - Exercicio 1: Soft-NMS
def soft_nms(boxes, scores, sigma=0.5, threshold=0.001):
    """
    Soft-NMS: em vez de remover duplicatas, reduz o score.
    scores_new[i] = scores[i] * exp(-IoU^2 / sigma)

    Args:
        boxes: lista de [x1, y1, x2, y2]
        scores: lista de confidence scores
        sigma: parametro de decaimento (menor = mais agressivo)
        threshold: score minimo para manter

    Returns:
        indices dos boxes mantidos, scores atualizados
    """
    keep = None  # TAREFA DO ALUNO: implementar
    new_scores = None  # TAREFA DO ALUNO: implementar
    return keep, new_scores

# Teste
test_boxes = [[10, 10, 100, 100], [20, 20, 110, 110], [200, 200, 300, 300]]
test_scores = [0.9, 0.85, 0.8]
# keep, new_scores = soft_nms(test_boxes, test_scores)

In [ ]:
# SOLUCAO - Exercicio 1: Soft-NMS
def soft_nms(boxes, scores, sigma=0.5, threshold=0.001):
    # Definir variaveis
    boxes = [list(b) for b in boxes]
    scores = list(scores)
    n = len(boxes)
    indices = list(range(n))
    new_scores = list(scores)

    keep = []
    while indices:
        # Encontrar max score
        max_idx = max(indices, key=lambda i: new_scores[i])
        keep.append(max_idx)
        indices.remove(max_idx)

        # Atualizar scores dos restantes
        to_remove = []
        for i in indices:
            iou = box_iou_manual(boxes[max_idx], boxes[i])
            # Gaussian decay
            new_scores[i] = new_scores[i] * np.exp(-(iou ** 2) / sigma)
            if new_scores[i] < threshold:
                to_remove.append(i)

        for i in to_remove:
            indices.remove(i)

    return keep, new_scores

# Comparacao Hard NMS vs Soft-NMS
test_boxes = [
    [10, 10, 100, 100],   # objeto 1
    [15, 15, 105, 105],   # duplicata proxima
    [20, 20, 110, 110],   # duplicata proxima
    [200, 200, 300, 300],  # objeto 2
]
test_scores = [0.95, 0.90, 0.85, 0.92]

# Hard NMS
hard_keep = nms_manual(test_boxes, test_scores, iou_threshold=0.5)
print('Hard NMS:')
for i in hard_keep:
    print(f'  Box {i}: score={test_scores[i]:.2f}')

print()

# Soft-NMS
soft_keep, soft_scores = soft_nms(test_boxes, test_scores, sigma=0.5)
print('Soft-NMS:')
for i in soft_keep:
    print(f'  Box {i}: score original={test_scores[i]:.2f}, novo={soft_scores[i]:.4f}')

print()
print('Diferenca: Hard NMS REMOVE boxes, Soft-NMS REDUZ scores')
print('Soft-NMS e melhor para objetos sobrepostos (ex: pessoas em multidao)')

### Exercicio 2: Calcular mAP Manual
Dado um conjunto de predicoes e ground truths, calcule mAP@0.5 manualmente.

Faca o seguinte:
- Ordene predicoes por confidence
- Para cada predicao, determine se e TP ou FP (IoU >= 0.5 com alguma GT nao usada)
- Calcule precision e recall cumulativos
- Compute AP como area sob a curva PR

In [ ]:
# PRATICA - Exercicio 2: Calcular mAP
def compute_map_manual(pred_boxes, pred_scores, pred_classes,
                       gt_boxes, gt_classes, iou_threshold=0.5):
    """
    Calcular mAP@0.5 manualmente.

    Args:
        pred_boxes: lista de [x1, y1, x2, y2]
        pred_scores: lista de confidence scores
        pred_classes: lista de class labels (int)
        gt_boxes: lista de [x1, y1, x2, y2]
        gt_classes: lista de class labels (int)
        iou_threshold: threshold para TP

    Returns:
        mAP (float), dict de AP por classe
    """
    mAP = None  # TAREFA DO ALUNO: implementar
    ap_per_class = None  # TAREFA DO ALUNO: implementar
    return mAP, ap_per_class

In [ ]:
# SOLUCAO - Exercicio 2: Calcular mAP
def compute_map_manual(pred_boxes, pred_scores, pred_classes,
                       gt_boxes, gt_classes, iou_threshold=0.5):
    classes = set(gt_classes)
    ap_per_class = {}

    for cls in classes:
        # Filtrar predicoes e GTs desta classe
        cls_pred_idx = [i for i, c in enumerate(pred_classes) if c == cls]
        cls_gt_idx = [i for i, c in enumerate(gt_classes) if c == cls]

        if not cls_gt_idx:
            ap_per_class[cls] = 0.0
            continue

        # Ordenar predicoes por score
        cls_pred_idx.sort(key=lambda i: pred_scores[i], reverse=True)

        n_gt = len(cls_gt_idx)
        gt_matched = [False] * n_gt

        precisions = []
        recalls = []
        tp_cum = 0
        fp_cum = 0

        for pi in cls_pred_idx:
            best_iou = 0
            best_gt = -1
            for gi_idx, gi in enumerate(cls_gt_idx):
                if gt_matched[gi_idx]:
                    continue
                iou = box_iou_manual(pred_boxes[pi], gt_boxes[gi])
                if iou > best_iou:
                    best_iou = iou
                    best_gt = gi_idx

            if best_iou >= iou_threshold and best_gt >= 0:
                tp_cum += 1
                gt_matched[best_gt] = True
            else:
                fp_cum += 1

            precisions.append(tp_cum / (tp_cum + fp_cum))
            recalls.append(tp_cum / n_gt)

        # AP = area sob curva PR (interpolacao 11 pontos)
        ap = 0.0
        for t in np.linspace(0, 1, 11):
            prec_at_recall = [p for p, r in zip(precisions, recalls) if r >= t]
            if prec_at_recall:
                ap += max(prec_at_recall) / 11.0
        ap_per_class[cls] = ap

    mAP = np.mean(list(ap_per_class.values())) if ap_per_class else 0.0
    return mAP, ap_per_class

# Teste
pred_boxes = [
    [28, 28, 128, 128],   # perto de GT 0
    [195, 175, 295, 275], # perto de GT 1
    [300, 50, 350, 100],  # falso positivo
    [35, 35, 135, 135],   # duplicata de GT 0
]
pred_scores = [0.95, 0.90, 0.80, 0.70]
pred_classes = [0, 1, 0, 0]

gt_boxes = [[30, 30, 130, 130], [200, 180, 300, 280]]
gt_classes = [0, 1]

mAP, ap_per_class = compute_map_manual(pred_boxes, pred_scores, pred_classes,
                                        gt_boxes, gt_classes, iou_threshold=0.5)

print(f'mAP@0.5 = {mAP:.4f}')
print()
for cls, ap in ap_per_class.items():
    print(f'  Classe {cls}: AP = {ap:.4f}')

print()
print('Nota: com mais predicoes e GTs, a curva PR seria mais suave')
print('Em producao, use COCO eval ou pycocotools para calculo oficial')

### Exercicio 3: Anchor Box Assignment
Dado um conjunto de anchors e ground truth boxes, atribua cada GT ao anchor
com maior IoU (para treino do detector).

Faca o seguinte:
- Para cada GT box, encontre o anchor com maior IoU
- Atribua labels: positivo (IoU >= 0.5), negativo (IoU < 0.3), ignorar (entre)
- Visualize os anchors positivos

In [ ]:
# PRATICA - Exercicio 3: Anchor Assignment
def assign_anchors(anchor_boxes, gt_boxes, pos_threshold=0.5, neg_threshold=0.3):
    """
    Atribuir labels para anchors baseado em IoU com GT boxes.

    Args:
        anchor_boxes: lista de [x1, y1, x2, y2]
        gt_boxes: lista de [x1, y1, x2, y2]
        pos_threshold: IoU minimo para positivo
        neg_threshold: IoU maximo para negativo

    Returns:
        labels: lista (1=positivo, 0=negativo, -1=ignorar)
        matched_gt: indice do GT mais proximo para cada anchor
    """
    labels = None  # TAREFA DO ALUNO: implementar
    matched_gt = None  # TAREFA DO ALUNO: implementar
    return labels, matched_gt

In [ ]:
# SOLUCAO - Exercicio 3: Anchor Assignment
def assign_anchors(anchor_boxes, gt_boxes, pos_threshold=0.5, neg_threshold=0.3):
    n_anchors = len(anchor_boxes)
    labels = [-1] * n_anchors  # default: ignorar
    matched_gt = [-1] * n_anchors

    for ai, anchor in enumerate(anchor_boxes):
        best_iou = 0
        best_gt = -1
        for gi, gt in enumerate(gt_boxes):
            iou = box_iou_manual(anchor, gt)
            if iou > best_iou:
                best_iou = iou
                best_gt = gi

        matched_gt[ai] = best_gt
        if best_iou >= pos_threshold:
            labels[ai] = 1  # positivo
        elif best_iou < neg_threshold:
            labels[ai] = 0  # negativo
        # else: -1 (ignorar)

    return labels, matched_gt

# Gerar anchors em grid 3x3 com 3 escalas
anchor_boxes = []
for cy in [50, 150, 250]:
    for cx in [50, 150, 250]:
        for size in [30, 60, 90]:
            anchor_boxes.append([cx-size//2, cy-size//2, cx+size//2, cy+size//2])

gt_boxes_test = [[40, 40, 120, 120], [180, 180, 280, 260]]

labels, matched_gt = assign_anchors(anchor_boxes, gt_boxes_test)

n_pos = labels.count(1)
n_neg = labels.count(0)
n_ign = labels.count(-1)

print(f'Total anchors: {len(anchor_boxes)}')
print(f'Positivos (IoU >= 0.5): {n_pos}')
print(f'Negativos (IoU < 0.3):  {n_neg}')
print(f'Ignorados (0.3-0.5):    {n_ign}')
print()

# Visualizar
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(-10, 310)
ax.set_ylim(-10, 310)
ax.set_aspect('equal')

# GT boxes
for gi, gt in enumerate(gt_boxes_test):
    rect = patches.Rectangle((gt[0], gt[1]), gt[2]-gt[0], gt[3]-gt[1],
                            linewidth=3, edgecolor='green', facecolor='green', alpha=0.15)
    ax.add_patch(rect)
    ax.text(gt[0], gt[1]-5, f'GT {gi}', fontsize=10, color='green', fontweight='bold')

# Anchors (colorizados por label)
for ai, (anchor, label) in enumerate(zip(anchor_boxes, labels)):
    if label == 1:
        color, alpha, lw = 'blue', 0.4, 2
    elif label == 0:
        color, alpha, lw = 'red', 0.1, 0.5
    else:
        color, alpha, lw = 'gray', 0.2, 0.5

    rect = patches.Rectangle((anchor[0], anchor[1]), anchor[2]-anchor[0], anchor[3]-anchor[1],
                            linewidth=lw, edgecolor=color, facecolor=color, alpha=alpha)
    ax.add_patch(rect)

# Legend
legend_elements = [
    patches.Patch(facecolor='green', alpha=0.3, label='Ground Truth'),
    patches.Patch(facecolor='blue', alpha=0.5, label=f'Positivo ({n_pos})'),
    patches.Patch(facecolor='red', alpha=0.3, label=f'Negativo ({n_neg})'),
    patches.Patch(facecolor='gray', alpha=0.3, label=f'Ignorado ({n_ign})'),
]
ax.legend(handles=legend_elements, fontsize=10, loc='upper right')
ax.set_title('Anchor Assignment: Azul=Positivo, Vermelho=Negativo', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('/tmp/anchor_assignment.png', dpi=100, bbox_inches='tight')
plt.show()

print('Na pratica, ratio positivo:negativo e ~1:3 (hard negative mining)')
print('Anchors ignorados nao contribuem para a loss (evita ruido)')

### O que observar sobre Metricas Alem do mAP

Alem de mAP, considere:
- **mAP-small/medium/large:** performance por tamanho de objeto (COCO reporta separado)
- **FPS (frames por segundo):** critico para real-time
- **Latencia p99:** pior caso de tempo de inferencia
- **Memory footprint:** quanto de GPU/RAM o modelo consome

### O que concluir sobre Reproducibilidade em Deteccao

Resultados de deteccao sao notoriamente dificeis de reproduzir. Pequenas mudancas em
hiperparametros (NMS threshold, confidence threshold, input resolution) podem mudar
mAP em +/- 2-3%. Sempre documente TAREFA DO ALUNOS os hiperparametros de avaliacao.

### Conexao com outros notebooks sobre Experimentacao

A importancia de documentar hiperparametros conecta com 1_5 (design de experimentos).
Um benchmark rigoroso de detectores precisa controlar: mesmo backbone, mesma resolucao,
mesmo NMS threshold, e mesmo numero de epocas de treino.

## 8. Erros Comuns e Armadilhas

### Erro 1: Confundir Formatos de Bounding Box
**Problema:** misturar `[x1, y1, x2, y2]` (corner) com `[xc, yc, w, h]` (center).
Um unico erro de conversao invalida TODAS as predicoes.
**Solucao:** definir uma convencao no inicio do projeto e usar funcoes de conversao
explicitas. Adicionar asserts: `assert x2 > x1` e `assert y2 > y1`.

### Erro 2: NMS antes do Confidence Threshold
**Problema:** aplicar NMS em todas as predicoes (incluindo as com confidence baixa).
Resultado: NMS demora 100x mais e pode criar artefatos.
**Solucao:** sempre filtrar por confidence primeiro (threshold=0.05-0.5), depois NMS.

### Erro 3: IoU Threshold Errado para Avaliacao
**Problema:** usar IoU=0.1 (muito permissivo) ou IoU=0.9 (muito rigoroso).
**Solucao:** reportar mAP@0.5 (PASCAL VOC) E mAP@[0.5:0.95] (COCO) para comparabilidade.

### Erro 4: Ignorar Objetos Pequenos
**Problema:** modelo tem 80% mAP geral mas 20% mAP-small.
**Solucao:** usar FPN, Mosaic augmentation, e multi-scale training.

### Erro 5: Nao Transformar Boxes durante Augmentation
**Problema:** aplicar flip/crop na imagem mas esquecer de transformar as bounding boxes.
**Solucao:** usar bibliotecas como Albumentations que transformam imagem + boxes atomicamente.

### Erro 6: Anchor Boxes Inadequadas
**Problema:** usar anchors padrao (ImageNet) em dataset com objetos de formas muito diferentes.
**Solucao:** rodar K-means nos aspect ratios do seu dataset para gerar anchors customizadas.
Ou usar detector anchor-free (FCOS, YOLOv8).

### Erro 7: Treinar sem Pre-treinamento
**Problema:** treinar detector do zero com dataset pequeno (< 5K imagens).
**Solucao:** SEMPRE usar backbone pre-treinado (ImageNet ou COCO). Fine-tuning
com 1-2K imagens ja funciona razoavelmente.

### O que observar sobre Deteccao em Tempo Real

Para aplicacoes real-time (> 30 FPS), considere:
1. **Input resolution:** reduzir de 640x640 para 320x320 quadruplica velocidade
2. **Backbone leve:** MobileNet ou ShuffleNet em vez de ResNet-101
3. **Quantizacao:** INT8 em vez de FP32 dobra throughput em GPU
4. **Batching:** processar multiplos frames simultaneamente

### O que concluir sobre a Escolha do Detector

Na pratica, a decisao segue esta heuristica:
- Real-time (> 30 FPS): YOLOv8 Nano/Small
- Melhor accuracy possivel: Cascade R-CNN + Swin Transformer
- Objetos muito pequenos: YOLOX ou RetinaNet com FPN P2-P6
- Instance segmentation: Mask R-CNN ou SOLOv2

### Conexao com outros notebooks sobre Deploy de Modelos

A escolha de detector impacta diretamente o deploy (6_1). Modelos menores permitem
edge deployment (celular, Raspberry Pi), enquanto modelos grandes requerem GPU na nuvem.
O trade-off speed/accuracy de 5A_1 (CNN fundamentos) se aplica diretamente aqui.

### Por que em ML: Deteccao como Fundamento para Tarefas Complexas

Deteccao e a base para:
- **Instance Segmentation** (Mask R-CNN): deteccao + mascara pixel-level
- **Pose Estimation** (keypoint detection): deteccao + pontos articulares
- **Tracking** (SORT, DeepSORT): deteccao frame-a-frame + associacao
- **Scene Understanding:** combinar deteccao + segmentacao + relacoes entre objetos

### O que observar sobre Datasets e Benchmarks

Os principais benchmarks de deteccao sao:
- **COCO** (80 classes, 330K imagens): padrao de fato, metricas rigorosas
- **PASCAL VOC** (20 classes, 11K imagens): mais simples, usado historicamente
- **Open Images** (600 classes, 9M imagens): maior e mais diverso
- **LVIS** (1200 classes, long-tail): testa generalizacao para classes raras

### O que concluir sobre Class Imbalance em Deteccao

Deteccao sofre de dois tipos de imbalance:
1. **Foreground vs Background:** 99%+ das posicoes nao tem objetos (Focal Loss resolve)
2. **Inter-class:** algumas classes tem 100x mais exemplos que outras

Para lidar: oversampling de classes raras, class-balanced loss, e augmentation focado.

### Conexao com outros notebooks sobre Avaliacao

Avaliacao de deteccao conecta com 2_4 (metricas). A curva PR de deteccao e analoga
a de classificacao, mas calculada por classe e por threshold de IoU. Entender precision
e recall de 2_4 e pre-requisito para entender mAP.

### Por que em ML: O Estado da Arte em 2024

A fronteira atual de deteccao inclui:
- **RT-DETR:** detector Transformer real-time sem NMS
- **YOLOv9:** Programmable Gradient Information para melhor treino
- **Grounding DINO:** deteccao open-vocabulary (descreva o que quer encontrar em texto)
- **SAM (Segment Anything):** segmentacao zero-shot que complementa deteccao

### O que observar sobre Deteccao 3D e Video

Deteccao nao se limita a imagens 2D estaticas:
- **3D Detection:** usa LiDAR ou depth cameras (PointPillars, CenterPoint)
- **Video Detection:** explora temporal consistency (TubeTK, TransVOD)
- **Multi-Modal:** combina camera + LiDAR + radar (BEVFusion)

### O que concluir sobre a Convergencia de Deteccao e Segmentacao

A tendencia moderna e unificar deteccao e segmentacao em um unico framework:
- Mask R-CNN ja faz deteccao + instance segmentation
- Panoptic Segmentation unifica semantic + instance segmentation
- SAM + Grounding DINO = deteccao + segmentacao open-vocabulary

### Conexao com outros notebooks sobre Segmentacao

O proximo notebook (5A_4) cobre segmentacao em detalhe. A relacao e direta:
deteccao encontra "onde" estao os objetos (boxes), segmentacao define "a forma exata"
de cada objeto (mascaras pixel-level). Mask R-CNN combina ambos.

### Por que em ML: Aplicacoes no Mundo Real

Deteccao de objetos e usada em:
- **Carros autonomos:** detectar pedestres, veiculos, sinalizacao
- **Medicina:** detectar tumores, celulas anomalas, fraturas
- **Varejo:** contagem de produtos, deteccao de furto
- **Agricultura:** detectar pragas, contar frutos, monitorar plantacoes
- **Seguranca:** deteccao de intrusos, reconhecimento de placas (ALPR)

## 9. Resumo e Conexoes

### Hierarquia de Conceitos

```
Deteccao de Objetos
├── Fundamentos
│   ├── Bounding Boxes (corner vs center format)
│   ├── IoU (metrica de sobreposicao)
│   └── mAP (metrica principal de avaliacao)
├── Arquiteturas
│   ├── Two-Stage (R-CNN -> Fast -> Faster -> Cascade)
│   ├── One-Stage (YOLO, SSD, RetinaNet)
│   └── Anchor-Free (FCOS, CenterNet, YOLOv8)
├── Componentes
│   ├── Feature Pyramid Network (multi-scale)
│   ├── NMS / Soft-NMS (pos-processamento)
│   └── Focal Loss (class imbalance)
└── Pratica
    ├── Data Augmentation (Mosaic, Copy-Paste)
    ├── Transfer Learning (backbone pre-treinado)
    └── Avaliacao (mAP@0.5, mAP@[0.5:0.95])
```

### Tabela de Conexoes

| Conceito | Conecta com | Relacao |
|----------|-------------|---------|
| IoU | 2_4 (metricas) | Precision/recall para deteccao |
| Backbone CNN | 5A_1 (CNN fundamentos) | Feature extraction compartilhada |
| Transfer Learning | 5A_2 (classificacao) | Fine-tuning do backbone |
| FPN multi-scale | 5A_1 (feature hierarquica) | Explora hierarquia de features |
| Segmentation | 5A_4 (segmentacao) | Mask R-CNN estende deteccao |
| Deploy | 6_1 (deploy) | Speed/accuracy trade-off |

### Checklist de Competencias

- [ ] Sei calcular IoU entre duas bounding boxes
- [ ] Entendo a diferenca entre two-stage e one-stage detectors
- [ ] Sei como NMS funciona e quando usar Soft-NMS
- [ ] Entendo o papel do FPN para deteccao multi-escala
- [ ] Sei interpretar mAP@0.5 e mAP@[0.5:0.95]
- [ ] Entendo como augmentation funciona em deteccao (boxes precisam ser transformados)

### Proximos Passos
- **5A_4 (Segmentacao):** de boxes para mascaras pixel-level
- **5A_5 (Vision Transformers):** DETR e deteccao com Transformers